# Creating Graphs from Building Topology with topologic_fast

This notebook demonstrates how to create topological graphs from building geometry.
It covers:

1. Creating building topology (CellComplex)
2. Building dual graphs from topology
3. Computing graph centrality measures
4. Visualizing topology and graphs together

**Note:** Direct IFC file parsing (`Graph.ByIFCPath`) is not yet implemented in topologic_fast.
This notebook demonstrates the graph creation concepts using programmatically created geometry.

## Import Required Libraries

In [ ]:
# Import topologic_fast
import topologic_fast as tf

# Import visualization libraries
import plotly.graph_objects as go
import numpy as np

print("topologic_fast imported successfully")
print("\nAvailable Graph methods:")
print("  - Graph.ByVerticesEdges()")
print("  - Graph.ByTopology()")
print("  - Graph.Vertices()")
print("  - Graph.Edges()")
print("  - Graph.AdjacencyMatrix()")
print("  - Graph.DegreeSequence()")
print("  - Graph.Density()")
print("  - Graph.Diameter()")

## Create a Multi-Room Building

We create a building with multiple rooms (cells) that share walls (faces).

In [ ]:
# Building parameters
room_width = 4.0
room_length = 5.0
room_height = 3.0

# Create a grid of rooms (3x2 layout, 2 floors)
rooms_x = 3
rooms_y = 2
num_floors = 2

cells = []
room_names = []

for floor in range(num_floors):
    for row in range(rooms_y):
        for col in range(rooms_x):
            x = col * room_width
            y = row * room_length
            z = floor * room_height
            
            cell = tf.Cell.Box(x, y, z, room_width, room_length, room_height)
            cells.append(cell)
            
            room_name = f"Room_F{floor}_R{row}_C{col}"
            room_names.append(room_name)

# Create CellComplex from cells
building = tf.CellComplex.ByCells(cells)

print(f"Building created:")
print(f"  Rooms per floor: {rooms_x * rooms_y}")
print(f"  Number of floors: {num_floors}")
print(f"  Total rooms: {len(cells)}")
print(f"  Total volume: {building.Volume():.2f} m^3")
print(f"\nRooms:")
for i, name in enumerate(room_names):
    print(f"  {i}: {name}")

## Create a Dual Graph from the CellComplex

The dual graph represents:
- **Vertices**: One for each Cell (room)
- **Edges**: Connect Cells that share a Face (wall/floor)

In [ ]:
# Create the dual graph using Graph.ByTopology
# This creates vertices at cell centroids and edges between cells that share faces
graph = tf.Graph.ByTopology(
    building,
    direct=True,
    via_shared_topologies=False,
    to_exterior_topologies=False,
    tolerance=0.001
)

print("Dual Graph created from CellComplex:")
print(f"  Vertices (rooms): {len(graph.Vertices())}")
print(f"  Edges (connections): {len(graph.Edges())}")

## Alternative: Manually Build the Adjacency Graph

For more control, we can manually determine adjacencies and build the graph.

In [ ]:
def cells_share_face(cell1, cell2, tolerance=0.1):
    """
    Check if two cells share a face by comparing face centroids.
    """
    faces1 = cell1.Faces()
    faces2 = cell2.Faces()
    
    for f1 in faces1:
        c1 = f1.CenterOfMass()
        for f2 in faces2:
            c2 = f2.CenterOfMass()
            dist = np.sqrt(sum((a - b) ** 2 for a, b in zip(c1, c2)))
            if dist < tolerance:
                return True
    return False

# Create vertices at cell centroids
vertices = []
for cell in cells:
    centroid = cell.CenterOfMass()
    v = tf.Vertex.ByCoordinates(centroid[0], centroid[1], centroid[2])
    vertices.append(v)

# Find adjacent cells and create edges
edges = []
adjacency = []

for i in range(len(cells)):
    for j in range(i + 1, len(cells)):
        if cells_share_face(cells[i], cells[j]):
            edge = tf.Edge.ByStartVertexEndVertex(vertices[i], vertices[j])
            edges.append(edge)
            adjacency.append((i, j, room_names[i], room_names[j]))

# Create graph
manual_graph = tf.Graph.ByVerticesEdges(vertices, edges)

print(f"Manual Graph:")
print(f"  Vertices: {len(manual_graph.Vertices())}")
print(f"  Edges: {len(manual_graph.Edges())}")
print(f"\nAdjacencies found:")
for i, j, name1, name2 in adjacency:
    print(f"  {name1} <-> {name2}")

## Compute Graph Centrality Measures

Centrality measures help identify important nodes (rooms) in the building.

In [ ]:
def compute_closeness_centrality(graph, tolerance=0.001):
    """
    Compute closeness centrality for each vertex.
    Closeness = 1 / (sum of shortest path distances to all other vertices)
    """
    vertices = graph.Vertices()
    n = len(vertices)
    centralities = []
    
    for i, v in enumerate(vertices):
        depth_map = graph.DepthMap(v, tolerance)
        # Sum of distances (excluding unreachable and self)
        total_dist = sum(d for d in depth_map if d > 0)
        reachable = sum(1 for d in depth_map if d > 0)
        
        if total_dist > 0 and reachable > 0:
            # Normalized closeness centrality
            closeness = reachable / total_dist
        else:
            closeness = 0.0
        
        centralities.append(closeness)
    
    return centralities

def compute_betweenness_centrality_simple(graph, tolerance=0.001):
    """
    Simplified betweenness centrality based on degree.
    (Full betweenness requires shortest path enumeration)
    """
    degree_seq = graph.DegreeSequence(tolerance)
    max_degree = max(degree_seq) if degree_seq else 1
    return [d / max_degree for d in degree_seq]

# Compute centralities
closeness = compute_closeness_centrality(manual_graph)
degree_centrality = compute_betweenness_centrality_simple(manual_graph)

print("Room Centrality Analysis:")
print("-" * 60)
print(f"{'Room':<20} {'Closeness':<15} {'Degree Norm':<15}")
print("-" * 60)

for i, name in enumerate(room_names):
    print(f"{name:<20} {closeness[i]:<15.4f} {degree_centrality[i]:<15.4f}")

# Find most central room
most_central_idx = closeness.index(max(closeness))
print(f"\nMost central room: {room_names[most_central_idx]}")

## Graph Properties

In [ ]:
# Compute various graph properties
print("Graph Properties:")
print("=" * 40)

# Density: ratio of edges to possible edges
density = manual_graph.Density()
print(f"Density: {density:.4f}")
print(f"  (0 = no edges, 1 = complete graph)")

# Diameter: longest shortest path
diameter = manual_graph.Diameter()
print(f"\nDiameter: {diameter}")
print(f"  (Maximum steps between any two rooms)")

# Degree sequence
degrees = manual_graph.DegreeSequence()
print(f"\nDegree Statistics:")
print(f"  Min degree: {min(degrees)}")
print(f"  Max degree: {max(degrees)}")
print(f"  Avg degree: {sum(degrees)/len(degrees):.2f}")

# Bipartite check
is_bipartite = manual_graph.IsBipartite()
print(f"\nIs Bipartite: {is_bipartite}")

# Complete check
is_complete = manual_graph.IsComplete()
print(f"Is Complete: {is_complete}")

## Adjacency Matrix

In [ ]:
# Get adjacency matrix
adj_matrix = manual_graph.AdjacencyMatrix()

print("Adjacency Matrix:")
print(f"(Row/Col indices correspond to room indices)")
print()

# Print header
header = "    " + " ".join([f"{i:3d}" for i in range(len(adj_matrix))])
print(header)
print("    " + "-" * (4 * len(adj_matrix)))

# Print matrix
for i, row in enumerate(adj_matrix):
    row_str = f"{i:3d}|" + " ".join([f"{v:3d}" for v in row])
    print(row_str)

## Visualize Building and Graph

In [ ]:
def create_cell_mesh(cell, offset_factor=0.95):
    """Create mesh data from a cell with slight inset for visibility."""
    centroid = cell.CenterOfMass()
    faces = cell.Faces()
    
    all_x, all_y, all_z = [], [], []
    all_i, all_j, all_k = [], [], []
    
    for face in faces:
        vertices = face.Vertices()
        if len(vertices) < 3:
            continue
        
        coords = [v.Coordinates() for v in vertices]
        offset = len(all_x)
        
        # Scale vertices towards centroid for visual separation
        for c in coords:
            x = centroid[0] + (c[0] - centroid[0]) * offset_factor
            y = centroid[1] + (c[1] - centroid[1]) * offset_factor
            z = centroid[2] + (c[2] - centroid[2]) * offset_factor
            all_x.append(x)
            all_y.append(y)
            all_z.append(z)
        
        for idx in range(1, len(coords) - 1):
            all_i.append(offset)
            all_j.append(offset + idx)
            all_k.append(offset + idx + 1)
    
    return (all_x, all_y, all_z), (all_i, all_j, all_k)

def visualize_building_with_graph(cells, room_names, manual_graph, closeness):
    """Create 3D visualization of building with dual graph overlay."""
    fig = go.Figure()
    
    # Normalize closeness for color mapping
    max_closeness = max(closeness)
    min_closeness = min(closeness)
    
    # Add cells (rooms)
    for i, cell in enumerate(cells):
        coords, indices = create_cell_mesh(cell)
        if not coords[0]:
            continue
        
        # Color by closeness centrality
        norm_centrality = (closeness[i] - min_closeness) / (max_closeness - min_closeness + 0.001)
        # Red for high centrality, blue for low
        color = f'rgb({int(255 * norm_centrality)}, 100, {int(255 * (1 - norm_centrality))})'
        
        fig.add_trace(go.Mesh3d(
            x=coords[0], y=coords[1], z=coords[2],
            i=indices[0], j=indices[1], k=indices[2],
            color=color,
            opacity=0.3,
            name=room_names[i],
            flatshading=True,
            showlegend=False
        ))
    
    # Add graph edges
    graph_edges = manual_graph.Edges()
    for edge in graph_edges:
        start_v, end_v = edge.Vertices()
        s = start_v.Coordinates()
        e = end_v.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[s[0], e[0]],
            y=[s[1], e[1]],
            z=[s[2], e[2]],
            mode='lines',
            line=dict(color='black', width=3),
            showlegend=False
        ))
    
    # Add graph vertices (room centroids)
    graph_vertices = manual_graph.Vertices()
    centers = [v.Coordinates() for v in graph_vertices]
    
    # Size by closeness centrality
    sizes = [10 + 20 * c / max_closeness for c in closeness]
    
    fig.add_trace(go.Scatter3d(
        x=[c[0] for c in centers],
        y=[c[1] for c in centers],
        z=[c[2] for c in centers],
        mode='markers+text',
        marker=dict(
            size=sizes,
            color=closeness,
            colorscale='RdYlBu_r',
            colorbar=dict(title='Closeness<br>Centrality'),
            line=dict(color='black', width=1)
        ),
        text=[f"{name}" for name in room_names],
        textposition='top center',
        textfont=dict(size=8),
        name='Room Centroids',
        hovertext=[f"{name}<br>Closeness: {c:.3f}" for name, c in zip(room_names, closeness)]
    ))
    
    # Update layout
    fig.update_layout(
        title='Building Graph Visualization<br>Color indicates closeness centrality (red = high, blue = low)',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.0)
            )
        ),
        width=900,
        height=700
    )
    
    return fig

# Create visualization
fig = visualize_building_with_graph(cells, room_names, manual_graph, closeness)
fig.show()

## Path Finding: Shortest Path Between Rooms

In [ ]:
def find_shortest_path(graph, start_idx, end_idx, room_names, tolerance=0.001):
    """
    Find shortest path between two rooms using BFS.
    """
    vertices = graph.Vertices()
    n = len(vertices)
    
    # BFS
    visited = [False] * n
    parent = [-1] * n
    queue = [start_idx]
    visited[start_idx] = True
    
    while queue:
        current = queue.pop(0)
        
        if current == end_idx:
            break
        
        # Get adjacent vertices
        adj_vertices = graph.AdjacentVertices(vertices[current], tolerance)
        
        for adj_v in adj_vertices:
            # Find index of adjacent vertex
            adj_coords = adj_v.Coordinates()
            for i, v in enumerate(vertices):
                v_coords = v.Coordinates()
                if all(abs(a - b) < tolerance for a, b in zip(adj_coords, v_coords)):
                    if not visited[i]:
                        visited[i] = True
                        parent[i] = current
                        queue.append(i)
                    break
    
    # Reconstruct path
    path = []
    current = end_idx
    while current != -1:
        path.append(current)
        current = parent[current]
    path.reverse()
    
    if path[0] != start_idx:
        return None  # No path found
    
    return [room_names[i] for i in path]

# Find paths between some rooms
print("Shortest Paths:")
print("=" * 40)

# From corner room to opposite corner
start_room = 0  # Room_F0_R0_C0
end_room = len(room_names) - 1  # Last room

path = find_shortest_path(manual_graph, start_room, end_room, room_names)
if path:
    print(f"\nPath from {room_names[start_room]} to {room_names[end_room]}:")
    print(f"  {' -> '.join(path)}")
    print(f"  Steps: {len(path) - 1}")

# Another example
start_room = 0
end_room = rooms_x * rooms_y  # First room of second floor

path = find_shortest_path(manual_graph, start_room, end_room, room_names)
if path:
    print(f"\nPath from {room_names[start_room]} to {room_names[end_room]}:")
    print(f"  {' -> '.join(path)}")
    print(f"  Steps: {len(path) - 1}")

## Visualize a Specific Path

In [ ]:
def visualize_path(cells, room_names, manual_graph, path_names):
    """Highlight a specific path in the building visualization."""
    fig = go.Figure()
    
    # Add all cells (faded)
    for i, cell in enumerate(cells):
        coords, indices = create_cell_mesh(cell)
        if not coords[0]:
            continue
        
        is_on_path = room_names[i] in path_names
        color = 'red' if is_on_path else 'lightgray'
        opacity = 0.5 if is_on_path else 0.1
        
        fig.add_trace(go.Mesh3d(
            x=coords[0], y=coords[1], z=coords[2],
            i=indices[0], j=indices[1], k=indices[2],
            color=color,
            opacity=opacity,
            name=room_names[i],
            flatshading=True,
            showlegend=is_on_path
        ))
    
    # Add path edges (highlighted)
    path_indices = [room_names.index(name) for name in path_names]
    graph_vertices = manual_graph.Vertices()
    
    for i in range(len(path_indices) - 1):
        start_v = graph_vertices[path_indices[i]]
        end_v = graph_vertices[path_indices[i + 1]]
        s = start_v.Coordinates()
        e = end_v.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[s[0], e[0]],
            y=[s[1], e[1]],
            z=[s[2], e[2]],
            mode='lines',
            line=dict(color='red', width=8),
            name=f"Step {i+1}",
            showlegend=True
        ))
    
    # Add path vertices
    path_centers = [graph_vertices[i].Coordinates() for i in path_indices]
    
    fig.add_trace(go.Scatter3d(
        x=[c[0] for c in path_centers],
        y=[c[1] for c in path_centers],
        z=[c[2] for c in path_centers],
        mode='markers+text',
        marker=dict(size=15, color='red', line=dict(color='black', width=2)),
        text=path_names,
        textposition='top center',
        name='Path Nodes'
    ))
    
    # Update layout
    fig.update_layout(
        title=f'Shortest Path: {path_names[0]} to {path_names[-1]}',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900,
        height=700
    )
    
    return fig

# Find and visualize a path
path = find_shortest_path(manual_graph, 0, len(room_names) - 1, room_names)
if path:
    fig = visualize_path(cells, room_names, manual_graph, path)
    fig.show()

## Summary

This notebook demonstrated graph creation from building topology:

1. **Building Topology**: Created a multi-room, multi-floor building as a CellComplex
2. **Dual Graph**: Created graphs where vertices represent rooms and edges represent adjacency
3. **Graph Analysis**: Computed centrality, density, diameter, and degree statistics
4. **Path Finding**: Implemented shortest path between rooms using BFS
5. **Visualization**: Created interactive 3D views with centrality coloring

### Features Not Yet Implemented in topologic_fast

- `Graph.ByIFCPath()` - Create graph directly from IFC file
- `Graph.ClosenessCentrality()` - Built-in closeness centrality
- `Graph.BetweennessCentrality()` - Betweenness centrality
- `Graph.ShortestPath()` - Built-in shortest path
- `Dictionary` storage on graph vertices for IFC properties

These features are planned for future releases.

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")